In [1]:
# Import necessary libraries

import pandas as pd

In [3]:
prompts_clients_df = pd.read_csv("data/prompts_clients.csv")
prompts_clients_df.head()

,client_id,county,sub_county,facility_name,gestation_weeks,parity,age,registration_date,interaction_date,interaction_type,...,referral_recommended,referral_destination,time_to_facility_hours,visited_facility,visit_outcome,delivery_outcome,mode_of_delivery,maternal_complication,client_experience_rating,experience_comment_theme
0,C00001,Kiambu,Ruiru,Ruiru Sub-County Hospital,14,0,34,2024-09-07,2024-11-03,danger_sign_alert,...,False,none,NaN,False,delivery,not_delivered_yet,SVD,none,NaN,respectful_care
1,C00002,Kiambu,Ruiru,Ruiru Health Centre,18,0,40,2024-01-28,2024-06-19,danger_sign_alert,...,True,Ruiru Sub-County Hospital,18.0,True,admitted,not_delivered_yet,NaN,none,4.0,lack_of_supplies
2,C00003,Machakos,Mavoko,Mavoko County Referral Hospital,11,2,43,2024-12-10,2025-01-05,education_message,...,False,none,NaN,False,referred_higher_level,not_delivered_yet,instrumental,none,NaN,respectful_care
3,C00004,Busia,Nambale,Nambale County Referral Hospital,24,2,34,2025-01-22,2025-02-11,danger_sign_alert,...,False,none,NaN,False,admitted,not_delivered_yet,SVD,none,NaN,respectful_care
4,C00005,Murang'a,Maragua,Maragua County Referral Hospital,30,5,42,2024-08-26,2024-09-20,education_message,...,False,none,NaN,False,referred_higher_level,not_delivered_yet,CS,none,NaN,lack_of_supplies


In [5]:
# Check of missing values

prompts_clients_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4500 entries, 0 to 4499
Data columns (total 21 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   client_id                 4500 non-null   str    
 1   county                    4500 non-null   str    
 2   sub_county                4500 non-null   str    
 3   facility_name             4500 non-null   str    
 4   gestation_weeks           4500 non-null   int64  
 5   parity                    4500 non-null   int64  
 6   age                       4500 non-null   int64  
 7   registration_date         4500 non-null   str    
 8   interaction_date          4500 non-null   str    
 9   interaction_type          4500 non-null   str    
 10  danger_sign_reported      4500 non-null   str    
 11  referral_recommended      4500 non-null   bool   
 12  referral_destination      4500 non-null   str    
 13  time_to_facility_hours    179 non-null    float64
 14  visited_facility   

###### There are missing values in time_to_facility_hours, mode_of_delivery, and client_experience_rating. These missing values occur because not all clients are referred to a facility, not all clients deliver during the PROMPTS period, and not all clients leave ratings after receiving services.

In [34]:
# 1 Check value ranges

# gestation_weeks

print(
    f"Gestation range values in the dataset is "
    f"{prompts_clients_df['gestation_weeks'].min()}-"
    f"{prompts_clients_df['gestation_weeks'].max()} weeks. "
    f"This is within the plausible range of 4-42."
)

# parity
print(f"The parity values in the dataset range from "
      f"{prompts_clients_df['parity'].min()} to "
      f"{prompts_clients_df['parity'].max()}, "
      f"which falls within the expected plausible range of 0 to 10 births."
      )

# age
print(f"The age range in the dataset is from "
      f"{prompts_clients_df['age'].min()} to "
      f"{prompts_clients_df['age'].max()}, "
      f"which falls within the expected plausible range of 10 to 55 years."
      )

# time_to_facility_hours
print(f"Time to facility in the dataset ranges from "
      f"{prompts_clients_df['time_to_facility_hours'].min()} to "
      f"{prompts_clients_df['time_to_facility_hours'].max()}, "
      f"which is plausible since negative values would be unrealistic and values above 72 hours would be considered extremely high."
      )

# client_experience_rating
print(f"The minimum rating recorded is " 
      f"{prompts_clients_df['client_experience_rating'].min()} and the maximum is "
      f"{prompts_clients_df['client_experience_rating'].max()}, "
      f"which lies within the expected 1–5 rating scale."
      )

Gestation range values in the dataset is 6-40 weeks. This is within the plausible range of 4-42.
The parity values in the dataset range from 0 to 5, which falls within the expected plausible range of 0 to 10 births.
The age range in the dataset is from 16 to 44, which falls within the expected plausible range of 10 to 55 years.
Time to facility in the dataset ranges from 1.0 to 48.0, which is plausible since negative values would be unrealistic and values above 72 hours would be considered extremely high.
The minimum rating recorded is 2.0 and the maximum is 5.0, which lies within the expected 1–5 rating scale.


In [37]:
# 2. Validate dates (registration < interaction) 

# Convert to datetime
prompts_clients_df["registration_date"] = pd.to_datetime(prompts_clients_df["registration_date"], errors="coerce")
prompts_clients_df["interaction_date"] = pd.to_datetime(prompts_clients_df["interaction_date"], errors="coerce")

# Check where the rule is violated: registration_date >= interaction_date
invalid_dates = prompts_clients_df[prompts_clients_df["registration_date"] > prompts_clients_df["interaction_date"]]

print(f"Number of records with invalid date order: {len(invalid_dates)}")
print(invalid_dates[["client_id", "registration_date", "interaction_date"]])

Number of records with invalid date order: 0
Empty DataFrame
Columns: [client_id, registration_date, interaction_date]
Index: []


###### All records have valid dates, with registration_date occurring before interaction_date for every client.

In [44]:
# 3. Check category consistency


# List of categorical columns
cat_cols = [
    "county",
    "sub_county",
    "facility_name",
    "interaction_type",
    "danger_sign_reported",
    "referral_recommended",
    "referral_destination",
    "visited_facility",
    "visit_outcome",
    "delivery_outcome",
    "mode_of_delivery",
    "maternal_complication",
    "experience_comment_theme",
]

# Unique values and counts for each categorical column
for col in cat_cols:
    print(f"\n==== {col} ====")
    print("Unique values:")
    print(prompts_clients_df[col].value_counts())



==== county ====
Unique values:
county
Bungoma     590
Murang'a    581
Kiambu      578
Busia       578
Machakos    560
Nairobi     558
Kakamega    535
Kisumu      520
Name: count, dtype: int64

==== sub_county ====
Unique values:
sub_county
Maragua        299
Teso South     296
Kimilili       296
Webuye         294
Kangundo       285
Nambale        282
Kandara        282
Mumias         277
Mavoko         275
Nyando         261
Kisumu East    259
Lurambi        258
Kasarani       210
Juja           202
Ruiru          200
Thika          176
Dagoretti      175
Embakasi       173
Name: count, dtype: int64

==== facility_name ====
Unique values:
facility_name
Nambale Health Centre                   160
Maragua Health Centre                   149
Mumias Health Centre                    147
Kangundo Health Centre                  147
Kandara Health Centre                   146
Webuye Health Centre                    145
Kimilili Health Centre                  143
Teso South Health Centre    

###### All values in the categorical columns are consistent, with no unexpected or invalid categories detected.